In [1]:
!pip install -q transformers torch scikit-learn pandas tqdm

In [6]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm

# パス設定（環境に合わせて調整）
data_path = "data/raw/full_emails.jsonl"
folds_path = "outputs/folds/common_folds.json"

# メールデータの読み込み
with open(data_path, "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]

# クラスラベルの文字列をID（整数）にマッピング
labels = sorted(list(set(item["label"] for item in raw_data)))
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

texts = [item["body_text"] for item in raw_data]
targets = [label2id[item["label"]] for item in raw_data]

# 共通Fold定義の読み込み
with open(folds_path, "r", encoding="utf-8") as f:
    folds_data = json.load(f) # 5-fold分のインデックスリスト

print(f"総データ数: {len(texts)} 件")
print(f"クラスラベル: {labels}")
print(f"Fold数: {len(folds_data)} 割")

総データ数: 800 件
クラスラベル: ['account_support', 'billing', 'product_inquiry', 'technical_issue']
Fold数: 2 割


In [12]:
def evaluate_predictions(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

TF-IDF + LinearSVC を使用した評価コードを追加します。

In [15]:
tfidf_results = []

# サンプルIDから元のインデックスへのマッピング
sample_id_to_original_index = {item['id']: i for i, item in enumerate(raw_data)}

# Fold データの再構築
num_folds = folds_data['metadata']['n_splits']
reconstructed_folds = []

for i in range(num_folds):
    current_train_indices = []
    current_val_indices = []
    for record in folds_data['records']:
        if record['fold_id'] == i:
            original_index = sample_id_to_original_index[record['sample_id']]
            if record['split_role'] == 'train':
                current_train_indices.append(original_index)
            elif record['split_role'] == 'validation':
                current_val_indices.append(original_index)
    reconstructed_folds.append({
        "train_indices": current_train_indices,
        "val_indices": current_val_indices
    })

# TF-IDF 評価の実行
for fold_idx, fold in enumerate(reconstructed_folds):
    train_idx = fold["train_indices"]
    val_idx = fold["val_indices"]

    X_train = [texts[i] for i in train_idx]
    y_train = [targets[i] for i in train_idx]
    X_val = [texts[i] for i in val_idx]
    y_val = [targets[i] for i in val_idx]

    vectorizer = TfidfVectorizer(max_features=5000)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_val_vec = vectorizer.transform(X_val)

    baseline_model = LinearSVC(random_state=42, max_iter=2000)
    baseline_model.fit(X_train_vec, y_train)
    preds = baseline_model.predict(X_val_vec)

    # すでに定義済みの evaluate_predictions を使用
    metrics = evaluate_predictions(y_val, preds)
    tfidf_results.append(metrics)

tfidf_df = pd.DataFrame(tfidf_results)
print("=== TF-IDF + LinearSVC (Cross-Validation Results) ===")
display(tfidf_df.agg(["mean", "std"]))

=== TF-IDF + LinearSVC (Cross-Validation Results) ===


,accuracy,precision,recall,f1_score
mean,0.609465,0.635210,0.609358,0.596279
std,0.121692,0.117975,0.124493,0.123999


In [17]:
class TextDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_len):
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        target = self.targets[item]

        # tokenizer.encode_plus ではなく tokenizer を直接呼び出す形式に修正
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(target, dtype=torch.long)
        }

def create_dataloader(texts, targets, tokenizer, max_len, batch_size):
    ds = TextDataset(texts, targets, tokenizer, max_len)
    return DataLoader(
        ds,
        batch_size=batch_size,
        num_workers=2
    )

### BERT 実装ステップ 1: モデルとトークナイザーの初期化

ここでは以下の処理を行います：
1. `distilbert-base-uncased` のトークナイザーとモデルのロード
2. 分類対象のクラス数（4）の設定
3. GPU (CUDA) が利用可能な場合はモデルを GPU へ転送

In [14]:
model_name = "distilbert-base-uncased"

# トークナイザーの初期化
tokenizer = AutoTokenizer.from_pretrained(model_name)

# モデルの初期化 (4クラス分類用)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels)
)

# デバイスの設定と転送
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"使用デバイス: {device}")
print(f"モデル '{model_name}' をロードし、{len(labels)} クラス用に初期化しました。")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


使用デバイス: cuda
モデル 'distilbert-base-uncased' をロードし、4 クラス用に初期化しました。


### BERT 実装ステップ 1: モデルとトークナイザーの初期化

ここでは以下の処理を行います：
1. `distilbert-base-uncased` のトークナイザーとモデルのロード
2. 分類対象のクラス数（4）の設定
3. GPU (CUDA) が利用可能な場合はモデルを GPU へ転送

In [13]:
model_name = "distilbert-base-uncased"

# トークナイザーの初期化
tokenizer = AutoTokenizer.from_pretrained(model_name)

# モデルの初期化 (4クラス分類用)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels)
)

# デバイスの設定と転送
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"使用デバイス: {device}")
print(f"モデル '{model_name}' をロードし、{len(labels)} クラス用に初期化しました。")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


使用デバイス: cuda
モデル 'distilbert-base-uncased' をロードし、4 クラス用に初期化しました。


### BERT 実装ステップ 2: 交差検証ループによる学習と評価

共通の Fold 定義に基づき、以下の手順で 5-Fold 交差検証を実施します：
1. 各 Fold ごとにモデルを初期化し、GPUへ転送
2. DataLoader の作成
3. エポック単位での学習 (Fine-tuning)
4. 検証データでの推論とメトリクス算出

In [25]:
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

bert_results = []
oof_records = []  # 監査証跡用に学習ループ内でOOFを収集

def train_epoch(model, data_loader, optimizer, device):
    model.train()
    losses = []
    for batch in tqdm(data_loader, desc="Training", leave=False):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
    return np.mean(losses)

def get_predictions(model, data_loader, device):
    model.eval()
    predictions = []
    real_values = []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs.logits, dim=1)
            predictions.extend(preds.cpu().tolist())
            real_values.extend(labels.cpu().tolist())
    return predictions, real_values

# BERT 交差検証ループ
for fold_idx, fold in enumerate(reconstructed_folds):
    print(f"\n--- Starting Fold {fold_idx} ---")

    train_idx = fold["train_indices"]
    val_idx = fold["val_indices"]

    train_loader = create_dataloader([texts[i] for i in train_idx], [targets[i] for i in train_idx], tokenizer, MAX_LEN, BATCH_SIZE)
    val_loader = create_dataloader([texts[i] for i in val_idx], [targets[i] for i in val_idx], tokenizer, MAX_LEN, BATCH_SIZE)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(labels)).to(device)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer, device)
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f}")

    # このFoldの最終的な予測を取得
    y_pred, y_true = get_predictions(model, val_loader, device)
    metrics = evaluate_predictions(y_true, y_pred)
    bert_results.append(metrics)

    # OOFレコードを保存（ここで各Foldの実際の予測を記録）
    for idx_in_val, (p, t) in enumerate(zip(y_pred, y_true)):
        original_idx = val_idx[idx_in_val]
        oof_records.append({
            "sample_id": raw_data[original_idx]['id'],
            "fold_id": fold_idx,
            "predicted_label": id2label[p],
            "true_label": id2label[t]
        })

    print(f"Fold {fold_idx} Results: {metrics}")

bert_df = pd.DataFrame(bert_results)
oof_df = pd.DataFrame(oof_records)
print("\n=== BERT (DistilBERT) Cross-Validation Results ===")
display(bert_df.agg(["mean", "std"]))


--- Starting Fold 0 ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 - Loss: 1.1917


Epoch 2/3 - Loss: 0.3298


...

Epoch 1/3 - Loss: 1.0746


Epoch 2/3 - Loss: 0.2404


Epoch 3/3 - Loss: 0.0481


Fold 4 Results: {'accuracy': 0.7650602409638554, 'precision': 0.8133096926713947, 'recall': 0.7361853832442067, 'f1_score': 0.7094847344750044}

=== BERT (DistilBERT) Cross-Validation Results ===


,accuracy,precision,recall,f1_score
mean,0.790899,0.827834,0.774643,0.752626
std,0.051697,0.069024,0.046574,0.055891


### 最終結果の比較

共通の Fold を使用した TF-IDF ベースラインと BERT (DistilBERT) の評価結果を比較します。

In [19]:
# 平均スコアの抽出
tfidf_summary = tfidf_df.mean().to_frame().T
tfidf_summary["model"] = "TF-IDF + LinearSVC"

bert_summary = bert_df.mean().to_frame().T
bert_summary["model"] = "DistilBERT (Fine-tuned)"

# 比較表の作成
comparison_summary = pd.concat([tfidf_summary, bert_summary], ignore_index=True)
comparison_summary = comparison_summary[["model", "accuracy", "precision", "recall", "f1_score"]]

print("=== Model Comparison Summary (Mean Scores over 5-Folds) ===")
display(comparison_summary)

=== Model Comparison Summary (Mean Scores over 5-Folds) ===


,model,accuracy,precision,recall,f1_score
0,TF-IDF + LinearSVC,0.609465,0.635210,0.609358,0.596279
1,DistilBERT (Fine-tuned),0.777944,0.809918,0.760606,0.738359


### プロジェクト監査用成果物の生成

プロジェクト規約に基づき、手動転記を避けるための機械可読成果物を出力します。

In [26]:
import hashlib
import platform
import transformers

# 1. 成果物の保存
bert_df.to_csv("bert_fold_metrics.csv", index_label="fold_id")
oof_df.to_csv("bert_oof_predictions.csv", index=False)

# 2. ハッシュと環境情報の取得
data_content = open(data_path, "rb").read()
data_hash = hashlib.sha256(data_content).hexdigest()

manifest = {
    "project_id": "task10",
    "target_data_hash": "53c6f8949a2c3c2c75351122e31dff6b43ca6ff8a4d8326947d387b75b9a0bbc",
    "actual_data_hash": data_hash,
    "model_config": {
        "model_name": model_name,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LEN,
        "random_seed": 42
    },
    "environment": {
        "transformers_version": transformers.__version__,
        "torch_version": torch.__version__,
        "python_version": platform.python_version(),
        "device": str(device)
    },
    "artifacts": [
        "bert_fold_metrics.csv",
        "bert_oof_predictions.csv",
        "execution_manifest.json"
    ]
}

with open("execution_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4, ensure_ascii=False)

print("整合性の取れた成果物ファイルが再生成されました。")
print(f"ハッシュ検証: {data_hash == manifest['target_data_hash']}")

整合性の取れた成果物ファイルが再生成されました。
ハッシュ検証: True


### 成果物ファイルのダウンロード
以下のコードを実行して、生成された監査用ファイルをローカルPCに保存します。

In [24]:
from google.colab import files

# ダウンロード対象のファイルリスト
artifact_files = [
    "bert_fold_metrics.csv",
    "bert_oof_predictions.csv",
    "execution_manifest.json"
]

for file_path in artifact_files:
    files.download(file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>